## 3 — Gráficos: Visão por Indicador


| Gráfico | Tipo | Descrição |
|--------|------|-----------|
| 11  | Barras agrupadas | Indicador selecionado por tipo de curso e ano |
| 12 | Barras horizontais | Ranking do indicador por curso (último ano) |
| 13  | Heatmap | Indicador por Curso × Ano |
| 14 | Heatmap | Todos os indicadores por Curso (último ano) |

#### 3.1 Importações, configurações e paletas

In [25]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sys
import os

sys.path.append(os.path.abspath(".."))

from utils import (
    carregar_dados,
    calcular_indicadores,
    gerar_mapa_cores,
    aplicar_layout_light,
    escala_indicador,
    CORES_INDICADORES,
    CORES_CATEGORIA,
    CORES_CONCLUSAO_LIGHT,
    CORES_FLUXO_LIGHT,
    CORES_MATRICULAS_ATIVAS_LIGHT,
    CORES_SITUACAO_LIGHT,
    COR_TEMPO_MEDIANO,
    CORES_CURSO,
    CORES_SITUACAO,
    PALETA_QUALITATIVA_LIGHT,
    ROTULOS_INDICADORES,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")



#### 3.2 Carga dos dados tratados

In [26]:
df_completo = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")
print("Shape:", df_completo.shape)
df_completo.head(3)

Shape: (10285, 18)


,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino


#### 3.3 Filtros utilizados e cálculo dos indicadores

Os filtros abaixo podem ser alterados manualmente para reproduzir cenários específicos.

In [27]:
periodo = (int(df_completo['Ano'].min()), int(df_completo['Ano'].max()))
tipos_selecionados = sorted(df_completo['Tipo de Curso'].dropna().unique())
cursos_selecionados = sorted(df_completo['Nome de Curso'].dropna().unique())

indicador = 'TR'  # alterar para: TC, TE, TR, IEf ou TPE

df = df_completo[
    (df_completo['Ano'] >= periodo[0])
    & (df_completo['Ano'] <= periodo[1])
    & (df_completo['Tipo de Curso'].isin(tipos_selecionados))
    & (df_completo['Nome de Curso'].isin(cursos_selecionados))
].copy()

ind_ano_curso = calcular_indicadores(df, ['Ano', 'Nome de Curso'])
ind_ano_tipo = calcular_indicadores(df, ['Ano', 'Tipo de Curso'])
ultimo_ano = int(ind_ano_curso['Ano'].max())
print('Último ano filtrado:', ultimo_ano)

Último ano filtrado: 2024


#### 3.4 Gráficos

#### Gráfico 11 — Indicador por tipo de curso e ano

Permite comparar se determinado indicador se concentra mais em algum tipo de curso ao longo do tempo.

In [28]:
cores_tipo = gerar_mapa_cores(ind_ano_tipo['Tipo de Curso'])
fig_g11 = px.bar(
    ind_ano_tipo, x='Ano', y=indicador, color='Tipo de Curso',
    barmode='group', text_auto='.1f', color_discrete_map=cores_tipo,
    labels={indicador: f'{ROTULOS_INDICADORES[indicador]} (%)', 'Tipo de Curso': ''},
)
fig_g11.update_xaxes(tickmode='linear', dtick=1)
aplicar_layout_light(fig_g11, altura=520)
fig_g11.show()

#### Gráfico 12 — Ranking do indicador por curso no último ano

Mostra quais cursos têm maior valor do indicador no último ano do período filtrado. Para Taxa de Evasão e Taxa de Retenção, valores altos indicam maior ponto de atenção; para Taxa de Conclusão, Indice de Eficiência e Taxa de Permanência e Êxito, valores altos indicam melhor desempenho.

In [29]:
ranking = (
    ind_ano_curso[ind_ano_curso['Ano'] == ultimo_ano][['Nome de Curso', indicador]]
    .sort_values(indicador, ascending=True)
)
fig_g12 = px.bar(
    ranking, x=indicador, y='Nome de Curso', orientation='h',
    color=indicador, color_continuous_scale=escala_indicador(indicador),
    text_auto='.1f', labels={indicador: f'{ROTULOS_INDICADORES[indicador]} (%)', 'Nome de Curso': ''},
)
fig_g12.update_layout(coloraxis_showscale=False)
aplicar_layout_light(fig_g12, altura=560)
fig_g12.show()

#### Gráfico 13 — Heatmap do indicador por Curso × Ano

In [30]:
pivot = ind_ano_curso.pivot_table(index='Nome de Curso', columns='Ano', values=indicador, aggfunc='mean')
fig_g13 = px.imshow(
    pivot, text_auto='.1f', aspect='auto', color_continuous_scale=escala_indicador(indicador),
    labels=dict(x='Ano', y='Curso', color=f'{ROTULOS_INDICADORES[indicador]} (%)'),
)
fig_g13.update_xaxes(tickmode='linear')
aplicar_layout_light(fig_g13, altura=620)
fig_g13.show()

#### Gráfico 14 — Heatmap dos indicadores por curso no último ano

In [31]:
indicadores = ['TC', 'TE', 'TR', 'IEf', 'TPE']
base_ultimo = ind_ano_curso[ind_ano_curso['Ano'] == ultimo_ano].copy()
heat_ind = base_ultimo.set_index('Nome de Curso')[indicadores].sort_index()
fig_g14 = px.imshow(
    heat_ind, text_auto='.1f', aspect='auto', color_continuous_scale='Purples',
    labels=dict(x='Indicador', y='Curso', color='Valor (%)'),
)
aplicar_layout_light(fig_g14, altura=620)
fig_g14.show()

In [32]:
# Maiores taxas de evasão e retenção no último ano
(
    base_ultimo[['Nome de Curso', 'TC', 'TE', 'TR', 'IEf', 'TPE', 'matr_atendidas']]
    .sort_values(['TE', 'TR'], ascending=False)
    .reset_index(drop=True)
)

,Nome de Curso,TC,TE,TR,IEf,TPE,matr_atendidas
0,Técnico em Informática,7.33,4.00,20.67,64.71,75.33,150
1,Técnico em Comércio,2.36,3.94,27.56,37.50,68.50,127
2,Técnico em Lazer,17.09,3.42,15.38,83.33,81.20,117
3,Eletrônica Industrial,2.72,3.40,42.86,44.44,53.74,147
4,Técnico em Eletrônica,3.66,2.44,35.98,60.00,61.59,164
5,Análise e Desenvolvimento de Sistemas,4.74,2.23,44.57,68.00,53.20,359
6,Processos Gerenciais,5.98,2.17,27.72,73.33,69.57,184
7,Letras - Português e Espanhol,3.11,1.86,43.48,62.50,54.04,161
8,Gestão Desportiva e de Lazer,1.77,0.88,48.67,66.67,50.44,113
9,Técnico em Agroecologia,6.85,0.68,29.45,90.91,69.86,146
